*Part 1.*

Implement Selection-, Insertion- and Merge sort in Python.

All three algorithms must be tested with comprehensive tests.

In [16]:
arrayX = [29,72,98,13,87]

def selection_sort(array: list) -> list:
    length = len(array)
    for i in range(length-1):
        min_val = i
        for j in range(i+1,length):
            if array[j] < array[min_val]:
                min_val = j
        if min_val != i:
            array[i], array[min_val] = array[min_val], array[i]

    return array

print("Selection sort:", selection_sort(arrayX))

def insertion_sort(array:list) -> list:
    for i in range(1, len(array)):
        key = array[i]
        j = i-1
        while j >= 0 and key < array[j]:
            array[j+1] = array[j]
            j -= 1
        array[j+1] = key

    return array

print("Insertion sort:", insertion_sort(arrayX))

def merge_sort(array:list) -> list:
    if len(array) <=1:
        return array

    mid = len(array) // 2
    left = merge_sort(array[:mid])
    right = merge_sort(array[mid:])

    return merge(left, right)

def merge(left: list, right: list) -> list:
    sorted = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            sorted.append(left[i])
            i += 1
        else:
            sorted.append(right[j])
            j += 1

    sorted.extend(left[i:])
    sorted.extend(right[j:])

    return sorted

print("Merge sort:", merge_sort(arrayX))

Selection sort: [13, 29, 72, 87, 98]
Insertion sort: [13, 29, 72, 87, 98]
Merge sort: [13, 29, 72, 87, 98]


In [17]:
import random
import unittest
from collections import namedtuple


class SortTests:
    """Shared tests run against every sorting algorithm. Subclasses set `sort`."""
    sort = None

    def check(self, data):
        self.assertEqual(type(self).sort(list(data)), sorted(data))

    # --- edge cases ---
    def test_empty_list(self):
        self.assertEqual(type(self).sort([]), [])

    def test_single_element(self):
        self.assertEqual(type(self).sort([42]), [42])

    def test_two_elements_sorted(self):
        self.check([1, 2])

    def test_two_elements_unsorted(self):
        self.check([2, 1])

    # --- ordering of input ---
    def test_already_sorted(self):
        self.check([1, 2, 3, 4, 5, 6, 7, 8, 9])

    def test_reverse_sorted(self):
        self.check([9, 8, 7, 6, 5, 4, 3, 2, 1])

    def test_unsorted(self):
        self.check([29, 72, 98, 13, 87])

    def test_odd_length(self):
        self.check([5, 3, 8, 1, 9, 2, 7])

    def test_even_length(self):
        self.check([5, 3, 8, 1, 9, 2])

    # --- values ---
    def test_duplicates(self):
        self.check([3, 1, 2, 3, 1, 2, 3])

    def test_all_equal(self):
        self.check([7, 7, 7, 7, 7])

    def test_negative_numbers(self):
        self.check([-3, 5, -1, 0, -10, 8])

    def test_floats(self):
        self.check([3.5, -1.25, 2.0, 0.0, 2.5])

    def test_mixed_int_and_float(self):
        self.check([3, 1.5, 2, 0.5, 1])

    def test_strings(self):
        self.check(["pear", "apple", "fig", "banana", "Apple"])

    def test_large_values(self):
        self.check([10**18, -10**18, 0, 10**9, -1])

    # --- result integrity ---
    def test_result_is_permutation_of_input(self):
        data = [4, 1, 3, 1, 5, 9, 2, 6]
        result = type(self).sort(list(data))
        self.assertEqual(len(result), len(data))
        self.assertCountEqual(result, data)

    def test_returns_list(self):
        self.assertIsInstance(type(self).sort([3, 1, 2]), list)

    # --- randomized comparison against Python's sorted() ---
    def test_random_lists(self):
        rng = random.Random(1234)
        for length in range(0, 50):
            for _ in range(5):
                data = [rng.randint(-100, 100) for _ in range(length)]
                with self.subTest(data=data):
                    self.check(data)

    def test_random_large_list(self):
        rng = random.Random(42)
        data = [rng.randint(-10_000, 10_000) for _ in range(1000)]
        self.check(data)

    def test_random_few_unique_values(self):
        rng = random.Random(7)
        data = [rng.randint(0, 3) for _ in range(200)]
        self.check(data)


Item = namedtuple("Item", ["key", "label"])
Item.__lt__ = lambda a, b: a.key < b.key
Item.__le__ = lambda a, b: a.key <= b.key
Item.__gt__ = lambda a, b: a.key > b.key


class StableSortTests:
    """Extra tests for stable algorithms: equal keys keep their original order."""

    def test_stable(self):
        data = [Item(2, "a"), Item(1, "b"), Item(2, "c"), Item(1, "d"), Item(2, "e")]
        result = type(self).sort(list(data))
        self.assertEqual([x.label for x in result], ["b", "d", "a", "c", "e"])


class InPlaceSortTests:
    """Extra tests for algorithms that sort the given list in place."""

    def test_sorts_in_place(self):
        data = [5, 2, 4, 1, 3]
        result = type(self).sort(data)
        self.assertIs(result, data)
        self.assertEqual(data, [1, 2, 3, 4, 5])


class TestSelectionSort(SortTests, InPlaceSortTests, unittest.TestCase):
    sort = staticmethod(selection_sort)


class TestInsertionSort(SortTests, InPlaceSortTests, StableSortTests, unittest.TestCase):
    sort = staticmethod(insertion_sort)


class TestMergeSort(SortTests, StableSortTests, unittest.TestCase):
    sort = staticmethod(merge_sort)

    def test_does_not_modify_input(self):
        data = [5, 2, 4, 1, 3]
        merge_sort(data)
        self.assertEqual(data, [5, 2, 4, 1, 3])


class TestMerge(unittest.TestCase):
    def test_both_empty(self):
        self.assertEqual(merge([], []), [])

    def test_one_empty(self):
        self.assertEqual(merge([1, 2, 3], []), [1, 2, 3])
        self.assertEqual(merge([], [1, 2, 3]), [1, 2, 3])

    def test_interleaved(self):
        self.assertEqual(merge([1, 3, 5], [2, 4, 6]), [1, 2, 3, 4, 5, 6])

    def test_all_left_smaller(self):
        self.assertEqual(merge([1, 2], [3, 4]), [1, 2, 3, 4])

    def test_all_right_smaller(self):
        self.assertEqual(merge([3, 4], [1, 2]), [1, 2, 3, 4])

    def test_different_lengths(self):
        self.assertEqual(merge([1], [0, 2, 3, 4]), [0, 1, 2, 3, 4])

    def test_duplicates_across_lists(self):
        self.assertEqual(merge([1, 2, 2], [2, 3]), [1, 2, 2, 2, 3])


unittest.main(argv=[""], verbosity=2, exit=False)


test_all_equal (__main__.TestInsertionSort.test_all_equal) ... ok
test_already_sorted (__main__.TestInsertionSort.test_already_sorted) ... ok
test_duplicates (__main__.TestInsertionSort.test_duplicates) ... ok
test_empty_list (__main__.TestInsertionSort.test_empty_list) ... ok
test_even_length (__main__.TestInsertionSort.test_even_length) ... ok
test_floats (__main__.TestInsertionSort.test_floats) ... ok
test_large_values (__main__.TestInsertionSort.test_large_values) ... ok
test_mixed_int_and_float (__main__.TestInsertionSort.test_mixed_int_and_float) ... ok
test_negative_numbers (__main__.TestInsertionSort.test_negative_numbers) ... ok
test_odd_length (__main__.TestInsertionSort.test_odd_length) ... ok
test_random_few_unique_values (__main__.TestInsertionSort.test_random_few_unique_values) ... ok
test_random_large_list (__main__.TestInsertionSort.test_random_large_list) ... ok
test_random_lists (__main__.TestInsertionSort.test_random_lists) ... ok
test_result_is_permutation_of_input 